<center><h1>Dong_Fengkai_HW3</h1></center>
<br>
<br>

Name: Fengkai Dong
<br>
Github Username: dfk0411
<br>
USC ID: 

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import pandas as pd
import numpy as np
import bootstrapped.bootstrap as bs
import bootstrapped.stats_functions as bs_stats
import os

Get the AReM Data Set

In [2]:
cols = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]
acts = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]
data = {}

for act in acts:
    data[act] = {}
    path = f"../data/AReM/{act}"
    for file in os.listdir(path):
        num = int(file.replace("dataset", "").replace(".csv", ""))
        if file.endswith(".csv"):
            df = pd.read_csv(
                f"{path}/{file}",
                comment="#",
                header=None,
                names=cols,
                sep=r"[,\s]+",
                engine="python",
                usecols=range(7)
            )
        data[act][num] = df

print(sum(len(data[act]) for act in acts))

88


### (b) Test and Train Data

In [3]:
train_data = {}
test_data = {}

for act in acts:
    train_data[act] = {}
    test_data[act] = {}
    for num, df in data[act].items():
        if act in ["bending1", "bending2"]:
            if num in [1, 2]:
                test_data[act][num] = df
            else:
                train_data[act][num] = df
        else:
            if num in [1, 2, 3]:
                test_data[act][num] = df
            else:
                train_data[act][num] = df

print("Training data:", sum(len(train_data[a]) for a in acts))
print("Test data:", sum(len(test_data[a]) for a in acts))

Training data: 69
Test data: 19


### (c) Feature Extraction

#### i. Research

Time-domain features used in classification include minimum, maximum, mean, median, standard deviation, variance, peak-to-peak value, first quartile, third quartile, and interquartile range.

#### ii. Extraction

In [4]:
series_cols = cols[1:]
feature_rows = []
instance = 1

for act in acts:
    for num in range(1, len(data[act])+1):
        df = data[act][num]
        row = {"Instance": instance}
        for i in range(len(series_cols)):
            col = series_cols[i]
            row[f"min{i+1}"] = df[col].min()
            row[f"max{i+1}"] = df[col].max()
            row[f"mean{i+1}"] = df[col].mean()
            row[f"median{i+1}"] = df[col].median()
            row[f"std{i+1}"] = df[col].std()
            row[f"1st quart{i+1}"] = df[col].quantile(0.25)
            row[f"3rd quart{i+1}"] = df[col].quantile(0.75)
        feature_rows.append(row)
        instance += 1

pd.DataFrame(feature_rows)

,Instance,min1,max1,mean1,median1,std1,1st quart1,3rd quart1,min2,max2,...,std5,1st quart5,3rd quart5,min6,max6,mean6,median6,std6,1st quart6,3rd quart6
0,1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,1.30,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
1,2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,1.22,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
2,3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,1.70,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
3,4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,3.00,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
4,5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,2.83,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,84,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,14.67,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
84,85,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,13.47,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
85,86,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,12.47,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350
86,87,23.50,46.25,34.873229,35.25,4.531720,31.75,38.2500,0.0,14.82,...,3.131076,13.7500,18.00,0.00,9.51,3.424646,3.27,1.690960,2.1700,4.5000


#### iii. Standard Deviation

In [5]:
features = pd.DataFrame(feature_rows)
results = []

for col in features.columns[1:]:
    values = features[col].values
    std = bs_stats.std(values)[0]
    ci = bs.bootstrap(
        features[col].values,
        stat_func=bs_stats.std,
        alpha=0.1
    )
    results.append({
        "Feature": col,
        "Standard Deviation": std,
        "Lower Bound": ci.lower_bound,
        "Upper Bound": ci.upper_bound
    })

pd.DataFrame(results)

,Feature,Standard Deviation,Lower Bound,Upper Bound
0,min1,9.515445,8.308212,10.810677
1,max1,4.369322,3.481310,5.418407
2,mean1,5.305314,4.764153,5.920904
3,median1,5.409056,4.846213,6.066046
4,std1,1.762056,1.586286,1.959363
5,1st quart1,6.118526,5.625454,6.698080
6,3rd quart1,5.109643,4.421654,5.920130
7,min2,0.000000,0.000000,0.000000
8,max2,5.033882,4.692320,5.466101
9,mean2,1.565194,1.435964,1.744827


#### iv. Select Features

Select mean, max, and std as the three most important time-domain features.

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic regression should have a lower training RSS than the linear regression, or at least no larger. The cubic model is more flexible, and under some circumstances the cubic model can degenerate into the linear model when the coefficients of $X^2$ and $X^3$ are zero. Therefore, adding these terms cannot increase the training RSS.

### (b) Linear Test

The linear regression should have a slightly lower test RSS. Since the true relationship is linear, the quadratic and cubic terms are unnecessary and may cause the cubic model to fit noise in the training data.

### (c) Not Linear Train

The cubic regression should have a lower training RSS. Regardless of the true relationship, the cubic model is more flexible than the linear model and can fit the training data better. Therefore, if the relationship is highly non-linear, the cubic regression could have a much lower training RSS; If the relationship is close to linear or cannot be well fitted by a cubic regression, we might expect similar RSS between two models.

### (d) Not Linear Testing

There is not enough information to tell which model will have a lower test RSS. Similar to the above, if the relationship is strongly nonlinear and is well approximated by a cubic function, the cubic model should perform better. If the relationship is only slightly nonlinear, the cubic model may have a similar or even lower test RSS.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 